In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
app_train_df = pd.read_csv("./data/application_train.csv")

In [ ]:
app_train_df["TARGET"].value_counts(normalize=True)

## DAYS_EMPLOYED weird value

In [ ]:
app_train_df["DAYS_EMPLOYED"].max()

In [ ]:
pd.reset_option("display.max_rows")

In [ ]:
max_days_employed = app_train_df.loc[app_train_df["DAYS_EMPLOYED"] == 365243]

In [ ]:
len(max_days_employed)

### Replace by NaN

In [ ]:
app_train_df.loc[app_train_df["DAYS_EMPLOYED"] == 365243, "DAYS_EMPLOYED"] = np.nan

In [ ]:
app_train_df.loc[app_train_df["DAYS_EMPLOYED"] == 365243]

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X = app_train_df.drop(["SK_ID_CURR", "TARGET"], axis=1)
y = app_train_df["TARGET"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Data for logistic regression

In [ ]:
numeric_cols = X_train.select_dtypes(include=[np.number]).columns.to_list()
cat_cols = X_train.columns.difference(numeric_cols).to_list()
cat_cols

### Missing Values counts

In [ ]:
app_train_df.isna().sum()

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline


In [ ]:
cat_pipeline = Pipeline([
    ("cat_imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore"))
])

num_pipeline = Pipeline([
    ("num_imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

ct = ColumnTransformer([
    ("cat", cat_pipeline, cat_cols),
    ("num", num_pipeline, numeric_cols)
])



In [ ]:
X_train_transformed = ct.fit_transform(X_train)
X_test_transformed = ct.transform(X_test)

In [ ]:
X_train_transformed.shape

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
logreg_pipe = Pipeline([
    ("preprocess", ct),
    ("model", LogisticRegression(max_iter=1000))
])

In [ ]:
logreg_pipe.fit(X_train, y_train)

In [ ]:
y_pred = logreg_pipe.predict(X_test)

In [ ]:
(y_pred == y_test).sum() / len(y_test)

In [ ]:
y_proba = logreg_pipe.predict_proba(X_test)

In [ ]:
y_proba = y_proba[:, 1]

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

fpr, tpr, thresholds = roc_curve(y_test, y_proba)

auc_score = roc_auc_score(y_test, y_proba)

print("ROC AUC:", auc_score)

# XGboost

In [ ]:
from xgboost import XGBClassifier

In [ ]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

In [ ]:
preprocess_xgb = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", "passthrough", numeric_cols)
])


In [ ]:
xgb_pipe = Pipeline([
    ("preprocess", preprocess_xgb),
    ("model", XGBClassifier(
        eval_metric="auc",
        random_state=42,
        n_jobs=-1
    ))
])

In [ ]:
from scipy.stats import randint, uniform, loguniform

In [ ]:
xgb_pipe.fit(X_train, y_train)

In [ ]:
y_proba = xgb_pipe.predict_proba(X_test)[:, 1]
test_auc = roc_auc_score(y_test, y_proba)

print("Test ROC AUC:", test_auc)